# Phase 6b: Dual-Path Latent-Space Analysis (English vs Parser/Wolfram)

Follow-up to Phase 6. Uses `AI-SecOps/latent_space_framework/redteam_kit` (same engine as `latent_space_redteaming.ipynb`).

## Two neural paths (both use Llama 3)

| Path | System prompt | What latent space shows |
|------|---------------|-------------------------|
| **English guardrail** | `config/english_policy_prompt.txt` | Policy judge representations → false allows, under-blocking |
| **Parser → Wolfram** | `config/parser_prompt.txt` | Semantic JSON parser representations → risk inflation, intent errors |

## What is NOT latent space

**Wolfram `rules.wl` is symbolic** — no neural activations after JSON is produced. You cannot run CKA on Wolfram Language rule evaluation. The closest proxy is comparing **parser-path** activations and **semantic JSON embeddings** against controls.

## Workflow

1. Locally: run Phase 6 on expanded dataset (`datasets/evaluation_prompts.jsonl`, 48 prompts)
2. `python -m eval.latent_space_runner --results results`
3. Upload `results/latent_space/cohort_*.jsonl` to Colab
4. Run this notebook (GPU). Download outputs to `results/latent_space/colab_output/`


In [ ]:
# Colab setup
# !pip install -q torch transformers accelerate scipy seaborn adversarial-robustness-toolbox

import json
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
DATA_DIR = Path("/content/wolfram_latent_space") if IN_COLAB else Path("../results/latent_space")
FRAMEWORK_DIR = Path("/content/latent_space_framework") if IN_COLAB else Path("../../AI SecOps/latent_space_framework")
OUTPUT_DIR = Path(f"/content/disagreement_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}") if IN_COLAB else Path("../results/latent_space/colab_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"  # align with Ollama llama3 judge
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
print("data:", DATA_DIR.resolve())
print("framework:", FRAMEWORK_DIR.resolve())


device: cuda
data: /content/wolfram_latent_space
framework: /content/latent_space_framework


In [ ]:
# Clone AI-SecOps (Colab) — includes latent_space_framework + Wolfram Guardrails
if IN_COLAB:
    get_ipython().system('git clone --depth 1 https://github.com/zbovaird/AI-SecOps.git /content/AI-SecOps')
    FRAMEWORK_DIR = Path("/content/AI-SecOps/latent_space_framework")
    GUARDRAILS_ROOT = Path("/content/AI-SecOps/Wolfram Guardrails")
else:
    GUARDRAILS_ROOT = Path("..")

sys.path.insert(0, str(FRAMEWORK_DIR / "redteam_kit"))
from core.modules.gradient_attacks import GradientAttackEngine
from core.modules.cka_analysis import CKAAnalyzer
from core.modules.latent_space_instrumentation import ModelInstrumentation

ENGLISH_POLICY_PROMPT = (GUARDRAILS_ROOT / "config/english_policy_prompt.txt").read_text(encoding="utf-8")
PARSER_PROMPT = (GUARDRAILS_ROOT / "config/parser_prompt.txt").read_text(encoding="utf-8")
print("loaded guardrail prompts from", GUARDRAILS_ROOT)


Cloning into '/content/AI-SecOps'...


remote: Enumerating objects: 380, done.
remote: Counting objects: 100% (380/380), done.
remote: Compressing objects: 100% (345/345), done.
remote: Total 380 (delta 32), reused 347 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (380/380), 1.64 MiB | 13.54 MiB/s, done.


Resolving deltas: 100% (32/32), done.


loaded guardrail prompts from /content/AI-SecOps/Wolfram Guardrails


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def load_cohorts(data_dir: Path) -> dict[str, list[dict]]:
    cohorts = {}
    for path in sorted(data_dir.glob("cohort_*.jsonl")):
        name = path.stem.replace("cohort_", "")
        cohorts[name] = load_jsonl(path)
    return cohorts


export_summary = json.loads((DATA_DIR / "export_summary.json").read_text(encoding="utf-8"))
cohorts = load_cohorts(DATA_DIR)
priority = export_summary.get("analysisPriority", [])
controls = cohorts.get("controls_agreement_correct", [])
print("priority:", priority)
print("cohort counts:", {k: len(v) for k, v in cohorts.items()})
print("controls:", len(controls))


priority: ['english_false_allow', 'english_under_block', 'english_policy_error', 'wolfram_rule_gap', 'wolfram_over_review']
cohort counts: {'controls_agreement': 6, 'controls_agreement_correct': 6, 'english_false_allow': 2, 'english_policy_error': 5, 'english_under_block': 2, 'wolfram_over_review': 1, 'wolfram_rule_gap': 1}
controls: 6


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# HF token required for Llama weights in Colab
# from huggingface_hub import login; login()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.eval()

instrumentation = ModelInstrumentation(model)
instrumentation.register_hooks()
engine = GradientAttackEngine(model, tokenizer, instrumentation=instrumentation)
cka = CKAAnalyzer()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
401 Client Error. (Request ID: Root=1-6a33517e-1e883b50486e9fef36b1f7f5;0e99f24b-c19a-4e24-b1ff-8118c79fd487)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Phase 0: baseline on agreement-correct controls
control_prompts = [row["prompt"] for row in controls]
baseline = engine.compute_baseline(control_prompts, num_variations=1)
targets = engine.identify_targets(baseline)

baseline_path = OUTPUT_DIR / "phase0_baseline.json"
baseline_path.write_text(json.dumps({
    "model": MODEL_NAME,
    "controlCount": len(control_prompts),
    "layerCount": len(baseline.layer_metrics),
    "targets": [t.layer_name for t in targets[:10]],
}, indent=2), encoding="utf-8")
print("baseline layers:", len(baseline.layer_metrics))
print("top targets:", [t.layer_name for t in targets[:5]])


NameError: name 'engine' is not defined

In [ ]:
# Dual-path forward passes: English policy judge vs Parser (Wolfram pipeline input)
# Reuses redteam_kit instrumentation — same hook pattern as latent_space_redteaming.ipynb


def chat_messages(system_prompt: str, user_prompt: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def capture_path_activations(system_prompt: str, user_prompt: str):
    messages = chat_messages(system_prompt, user_prompt)
    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"System: {system_prompt}\nUser: {user_prompt}\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    instrumentation.clear_captured()
    with torch.no_grad():
        model(**inputs)
    return instrumentation.get_captured_activations()


def mean_layer_vector(activations: dict) -> dict[str, torch.Tensor]:
    return {name: tensor.mean(dim=0) for name, tensor in activations.items()}


# Compare paths on all disagreement rows (dedupe by id)
seen = set()
all_rows = []
for name in priority:
    for row in cohorts.get(name, []):
        if row["id"] not in seen:
            seen.add(row["id"])
            all_rows.append(row)

path_divergence = []
for row in all_rows:
    eng = mean_layer_vector(capture_path_activations(ENGLISH_POLICY_PROMPT, row["prompt"]))
    par = mean_layer_vector(capture_path_activations(PARSER_PROMPT, row["prompt"]))
    layer_scores = {}
    for layer in eng:
        if layer in par:
            layer_scores[layer] = float(cka.compute_cka(eng[layer].unsqueeze(0), par[layer].unsqueeze(0)))
    path_divergence.append({
        "id": row["id"],
        "cohort": row.get("cohort"),
        "semanticRisk": row.get("semanticRisk"),
        "englishDecision": row.get("englishDecision"),
        "wolframDecision": row.get("wolframDecision"),
        "meanPathCka": sum(layer_scores.values()) / max(len(layer_scores), 1),
        "layerPathCka": layer_scores,
    })

path_path = OUTPUT_DIR / "english_vs_parser_path_cka.json"
path_path.write_text(json.dumps(path_divergence, indent=2), encoding="utf-8")
print("dual-path rows:", len(path_divergence))
print("wrote", path_path)


NameError: name 'tokenizer' is not defined

## Symbolic Wolfram proxy (not neural latent space)

`rules.wl` evaluates JSON with predicates — no embeddings inside Wolfram Engine.

**Proxies we can measure in Colab:**
1. **Parser-path activations** (cell above) — where semantic JSON is formed
2. **Semantic JSON text embeddings** — embed `json.dumps({intent,target,techniques,risk,notes})` and compare cohorts
3. **Rule outcome labels** — join `wolframDecision` / `triggeredRules` from Phase 6 exports (metadata only)

Low English-vs-parser CKA on a disagreement row suggests the two guardrail paths *represent the prompt differently* before any Wolfram rule fires.


In [ ]:
# Semantic JSON embedding proxy for Wolfram-path inputs
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np

    st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def semantic_json_text(row: dict) -> str:
        payload = {
            "intent": row.get("semanticIntent") or "unknown",
            "risk": row.get("semanticRisk"),
            "category": row.get("category"),
        }
        return json.dumps(payload, sort_keys=True)

    control_json_vecs = st_model.encode([semantic_json_text(r) for r in controls])
    control_centroid = np.mean(control_json_vecs, axis=0)

    json_embedding_results = []
    for row in all_rows:
        vec = st_model.encode([semantic_json_text(row)])[0]
        cos = float(np.dot(vec, control_centroid) / (np.linalg.norm(vec) * np.linalg.norm(control_centroid) + 1e-9))
        json_embedding_results.append({
            "id": row["id"],
            "cohort": row.get("cohort"),
            "semanticRisk": row.get("semanticRisk"),
            "wolframDecision": row.get("wolframDecision"),
            "cosineToControlSemanticJson": cos,
        })

    json_path = OUTPUT_DIR / "semantic_json_embedding_proxy.json"
    json_path.write_text(json.dumps(json_embedding_results, indent=2), encoding="utf-8")
    print("wrote", json_path)
except ImportError:
    print("sentence-transformers not installed; skip JSON embedding proxy")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

wrote /content/disagreement_analysis_20260618_020104/semantic_json_embedding_proxy.json


In [ ]:
# Per-cohort CKA vs controls — path-aware
# English cohorts: english_policy system prompt. Wolfram cohorts: parser system prompt.

WOLFRAM_COHORTS = {"wolfram_rule_gap", "wolfram_over_review"}


def system_for_cohort(cohort_name: str) -> str:
    return PARSER_PROMPT if cohort_name in WOLFRAM_COHORTS else ENGLISH_POLICY_PROMPT


control_vectors = [
    mean_layer_vector(capture_path_activations(ENGLISH_POLICY_PROMPT, p))
    for p in control_prompts
]
control_ref = {k: torch.stack([v[k] for v in control_vectors]).mean(dim=0) for k in control_vectors[0]}

cohort_results = {}
for cohort_name in priority:
    rows = cohorts.get(cohort_name, [])
    if not rows:
        continue
    sys_prompt = system_for_cohort(cohort_name)
    layer_cka = {}
    for row in rows:
        act = mean_layer_vector(capture_path_activations(sys_prompt, row["prompt"]))
        for layer, vec in act.items():
            if layer not in control_ref:
                continue
            score = cka.compute_cka(control_ref[layer].unsqueeze(0), vec.unsqueeze(0))
            layer_cka.setdefault(layer, []).append(float(score))
    cohort_results[cohort_name] = {
        "count": len(rows),
        "analysisPath": "parser" if cohort_name in WOLFRAM_COHORTS else "english",
        "rows": rows,
        "meanCkaByLayer": {layer: sum(vals) / len(vals) for layer, vals in layer_cka.items()},
    }

results_path = OUTPUT_DIR / "cohort_cka_vs_controls.json"
results_path.write_text(json.dumps(cohort_results, indent=2, default=str), encoding="utf-8")
print("wrote", results_path)


NameError: name 'tokenizer' is not defined

In [ ]:
# English-path gradient attacks (redteam_kit Phase 2-4) on policy failures
# Uses english_policy system context via raw prompt — same engine as latent_space_redteaming.ipynb

sensitive_rows = []
for name in ("english_false_allow", "english_policy_error"):
    for row in cohorts.get(name, []):
        sensitive_rows.append(row)

attack_results = []
for row in sensitive_rows[:12]:
  instrumentation.clear_captured()
  eng_act = capture_path_activations(ENGLISH_POLICY_PROMPT, row["prompt"])
  result = engine.attack_with_evaluation(row["prompt"], targets)
  attack_results.append({
      "id": row["id"],
      "cohort": row.get("cohort"),
      "analysisPath": "english",
      "semanticRisk": row.get("semanticRisk"),
      "expectedDecision": row.get("expectedDecision"),
      "englishDecision": row.get("englishDecision"),
      "wolframDecision": row.get("wolframDecision"),
      "attack": str(result),
  })

attack_path = OUTPUT_DIR / "english_path_gradient_attacks.json"
attack_path.write_text(json.dumps(attack_results, indent=2, default=str), encoding="utf-8")
print("english-path attacks:", len(attack_results))


NameError: name 'instrumentation' is not defined

## Interpretation guide

| Cohort | Latent-space question |
|--------|----------------------|
| `english_false_allow` | Do prompts look like benign controls in embedding space while parser assigns high risk? |
| `english_policy_error` | Are there steerable basins where policy judge drifts ALLOW despite risky semantics? |
| `wolfram_rule_gap` / `wolfram_over_review` | Is parser risk inflated on educational prompts (activation stability vs controls)? |

## Next steps after Colab

1. Download `OUTPUT_DIR` artifacts back to `results/latent_space/colab_output/`
2. Re-run `python -m eval.latent_space_runner --results results` locally to merge embedding probe
3. Expand `datasets/evaluation_prompts.jsonl` before investing in larger GPU runs (N=12 is exploratory)

Reference methodology: `AI-SecOps/latent_space_framework/notebooks/latent_space_redteaming.ipynb` (Cells 26–32).
